# Phishing Detection and Training System: A Generative AI Solution

## The Problem
Phishing attacks are one of the most persistent and effective cyber security threats everyone faces today. Traditional rule-based detection systems struggle with the following issues:

- Evolving phishing tactics
- Sophisticated impersonation techniques
- High false-positive rates
- Limited ability to generate training materials based on actual word attacks
- Issues with adaptation to new attack patterns without manual updates

## How Generative AI Can Solve This Challenge
This notebook demonstrates a solution using gen AI to detect, analyse and train users against phishing attempts by:

1. Language Understanding: Using LLM to analyse email content beyond simple pattern matching.
2. Domain Analysis: Employing AI agents with tools that allow it to investigate sender domain and embedded URLs.
3. Semantic Similarity Detection: Using vector embeddings DB to identify emails similar to already known phishing attempts.
4. Report Generation: Generates report for each email analysed and the verdict with score how likely the email is phishing.
5. Automated Training Generation: At the end of the recognition cycle, educational materials based on actual phishing are created.

## What Does Code Contain?
The notebook walks through:
1. Creating a vector store of emails for similarity detection. 
2. Building an LLM-powered email content analyzer.
3. Implementing an AI agent for domain investigation.
4. Combining multiple analysis approaches for comprehensive threat assessment.
5. Generating report for each of analysed emails.
6. Generating targeted training materials from detected phishing techniques.

Each section includes code and an explanation of how Gen-AI capabilities are applied to produce effective, adaptable security solution.

## Use case description

### Email Scanning and Detection
* System scans provided emails for phishing indicators
* Analyzes a body of the message domain of the sender and links and validates its reputation
* Saves already found phishing attempts into Vector DB for retrieval for further use as an example or fine-tuning
* Flags suspicious emails as phishing with confident score.
* Generates a report for each of the email

### Traning Guide Creation
* Creates training document with an explanation of phishing techniques used in scanned emails, examples of this technique, identification steps, action items and Q&A.

### Benefits
* Reduced successful phishing attacks
* Improved security awareness across the organization
* Data-drive approach to security training resources
* Security culture improvement

## Implementation

### Setup

In [1]:
# Install required dependencies
!pip install -qU "langchain-chroma>=0.1.2"
!pip install whois
!pip install -U langgraph
!pip install langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mysql-connector-python 8.0.32 requires protobuf<=3.20.3,>=3.11.0, but you have protobuf 5.29.4 which is incompatible.


   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.4 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   --------- ------------------------------ 1.0/4.3 MB 5.0 MB/s eta 0:00:01
   ----------------- ---------------------- 1.8/4.3 MB 4.8 MB/s eta 0:00:01
   ----------------------------- ---------- 3.1/4.3 MB 5.0 MB/s eta 0:00:01
   ------------------------------------ --- 3.9/4.3 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 4.3/4.3 MB 4.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Uninstalling protobuf-5.29.4:
      Successfully uninstalled protobuf-5.29.4
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.71.0
    Uninstalling grpcio-1.71.0:
      Successfully uninstalled grpcio-1.71.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mysql-connector-python 8.0.32 requires protobuf<=3.20.3,>=3.11.0, but you have protobuf 6.31.0rc1 which is incompatible.
opentelemetry-proto 1.32.1 requires protobuf<6.0,>=5.0, but you have protobuf 6.31.0rc1 which is incompatible.


In [5]:
# Import Kaggle specific APIs
import kagglehub
from kagglehub import KaggleDatasetAdapter
from kaggle_secrets import UserSecretsClient

# Import general use APIs
import os
import re
import whois
import dns.resolver
import pandas as pd
from pandas import DataFrame, Series
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, TypedDict, Dict, Annotated, Tuple, Generic, TypeVar, Optional, Union, Any
from pydantic import BaseModel, Field
from datetime import datetime
from dataclasses import dataclass
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import time
from datetime import datetime

ModuleNotFoundError: No module named 'kaggle_secrets'

In [ ]:
# Import LangChain APIs
from langchain.llms.base import LLM
from langchain.chains.base import Chain
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.tools import tool
from langchain_core.embeddings import Embeddings
from langchain.docstore.document import Document
from langchain.chains import LLMChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma 
from langchain.agents.agent import AgentExecutor
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# Import LangGraph APIs
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

In [ ]:
# Extract secret from Kaggle Env
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
# Setup secret so that it could be consumed by LangChain Google GenAI Models
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
# Load Kaggle phishing emails dataset as an example data to be processed
# This could by any other data that company or researcher want to analyse
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "naserabdullahalam/phishing-email-dataset",
  "CEAS_08.csv"
)

### Vector store for similar cases retrival

Such a database should be used as an archive of previous emails that were identified as phishing, as samples possess both commercial and educational value, they can be retrieved easily by doing similarity search.

These could be used for further fine-tuning of the model and as a way to show more examples of the possible real-world phishing attempts for the user in a step that generates the training material for employees.

In [ ]:
def create_email_vector_store(df: DataFrame, embedding:Embeddings) -> Chroma:    
    """
    Create a vector store using email content from a DataFrame.
    
    This function processes email data from a DataFrame, creates document objects
    with formatted content and metadata, and stores them in a Chroma vector store
    for semantic search capabilities.
    
    Args:
        df (DataFrame): A pandas DataFrame containing email data with columns:
                       'subject', 'body', 'sender', 'receiver', and 'date'.
        embedding (Embeddings): The embedding model to use for vectorizing documents.
    
    Returns:
        Chroma: A Chroma vector store containing the processed email documents.
    
    Example:
        >>> email_df = pd.read_csv('emails.csv')
        >>> vector_store = create_email_vector_store(email_df, OpenAIEmbeddings())
    """
    documents: List[Document] = []
    
    for idx, row in df.iterrows():
        # Formatting of subject and body into single string for easier storage and better search capabilities
        content = f"SUBJECT: {row['subject']}\n\nBODY: {row['body']}"

        # Additional metadata is stored for further reference
        metadata = {
            "sender": row["sender"],
            "receiver": row["receiver"],
            "date": row["date"],
            "id": idx
        }

        # Push preprocessed email into temporary array
        documents.append(Document(
            page_content=content,
            metadata=metadata
        ))
        
    # Create vector store
    vector_store = Chroma.from_documents(documents=documents, embedding=embedding)
    
    return vector_store

In [ ]:
@dataclass
class SimilarEmail:
    """Represents an email similar to a query with its similarity score."""
    content: str
    metadata: Dict[str, Any]
    similarity_score: float

def find_similar_phishing_emails(
    vector_store: Chroma, 
    subject: str, 
    body: str, 
    k: int = 2
) -> List[SimilarEmail]:
    """
    Find similar emails to the input email

    Parameters
    vector_store: Chroma
        The Chroma vector database containing embedded email documents to search within
    subject: str
        The subject of email to match against
    body: str
        The email body content to match against
    k:int, default=5

    Returns
    List[SimilarEmail]
        A list of dictionaries containing:
        - content: Truncated preview of the email content (first 200 chars)
        - metadata: Original metadata associated with the email document
        - similarity_score: Numerical similarity score (lower is more similar)
        
    Example:
        >>> vector_store = create_email_vector_store(email_df, embedding)
        >>> similar_emails = find_similar_phishing_emails(
        ...     vector_store, 
        ...     "Password Reset Required", 
        ...     "Please click the link to reset your password"
        ... )
    """

    # Replicates the structure of content that was used to create embeddings in the database
    query = f"SUBJECT: {subject}\n\nBODY: {body}"
    results = vector_store.similarity_search_with_score(query, k=k)

    # Map found documents to a class
    return [SimilarEmail(
        content=doc[0].page_content,
        metadata=doc[0].metadata,
        similarity_score=doc[1]
    ) for doc in results]

### Email Analysis

#### Email content analysis

In [ ]:
class PhishingAnalysisResult(BaseModel):
    """Result of email body phishing analysis detection."""
    phishing_score: int = Field(description="Phishing confidence score from 0-10")
    suspicious_elements: List[str] = Field(description="List of suspicious elements in the email")
    manipulation_techniques: List[str] = Field(description="Manipulation techniques used in the email")
    recommendations: List[str] = Field(description="Recommendations for recipients")
    is_phishing: bool = Field(description="Overall determination if email is phishing")

def setup_email_analysis_chain(llm: LLM) -> Chain:    
    """
    Creates and returns a chain for analyzing emails for phishing indicators.
    
    This function sets up an LLM chain that takes an email subject and body,
    analyzes it for phishing characteristics, and returns structured results
    with confidence score, suspicious elements, manipulation techniques,
    recommendations, and an overall phishing determination.
    
    Args:
        llm (BaseLLM): The language model to use for email analysis
    
    Returns:
        LLMChain: A chain that takes email subject and body as input and 
                 returns a structured PhishingAnalysisResult
    
    Example:
        >>> model = ChatOpenAI(temperature=0)
        >>> analysis_chain = setup_email_analysis_chain(model)
        >>> result = analysis_chain.invoke({
        ...     "subject": "Urgent: Account Compromise",
        ...     "body": "Click here to verify your account immediately!"
        ... })
        >>> print(f"Phishing score: {result.phishing_score}/10")
    """
    
    email_analysis_prompt = PromptTemplate(
        input_variables=["subject", "body"],
        template="""
        Analyze this email for signs of phishing. Consider both subject and body content.
        
        SUBJECT: {subject}
        
        BODY: {body}
        
        Provide an analysis with these specific points:
        1. Phishing confidence score (0-10)
        2. Key suspicious elements detected
        3. Manipulation techniques used
        4. Recommendations for recipients (3-5 MAX)
        5. Whether this is likely a phishing email (true/false)
        
        Ensure your response is valid JSON that can be parsed into the PhishingAnalysis model.
        """
    )

    email_analysis_chain = (
        email_analysis_prompt 
        | llm.with_structured_output(PhishingAnalysisResult)
    )

    return email_analysis_chain

In [ ]:
def analyze_email_content(
    row: Union[Series, dict], 
    email_analysis_chain: Chain
) -> Optional[PhishingAnalysisResult]:
    """
    Analyze an email for phishing indicators using the provided analysis chain.
    
    This function extracts the subject and body from the provided row data,
    sends it through the email analysis chain, and returns the structured
    phishing analysis result.
    
    Args:
        row (Union[Series, dict]): A pandas Series or dictionary containing 
                                  at minimum 'subject' and 'body' fields
        email_analysis_chain (LLMChain): The LangChain chain configured to 
                                        analyze emails for phishing indicators
    
    Returns:
        Optional[PhishingAnalysisResult]: The structured analysis result containing 
                                         phishing score, suspicious elements, etc.,
                                         or None if analysis failed
    
    Example:
        >>> df = pd.read_csv('emails.csv')
        >>> analysis_chain = setup_email_analysis_chain(llm)
        >>> for _, row in df.iterrows():
        ...     result = analyze_email_content(row, analysis_chain)
        ...     if result and result.is_phishing:
        ...         print(f"Phishing email detected: {row['subject']}")
    """    
    try:
        analysis = email_analysis_chain.invoke({
            "subject": row['subject'],
            "body": row['body']
        })
            
        return analysis
    except Exception as e:
        print(f"Error analyzing email: {e}")
        return None

#### Email sender domain analysis

##### Tools definition

In [ ]:
@tool
def extract_urls_from_email(email_body: str) -> List[str]:
    """
    Extract all URLs from an email body.
    
    This function uses regex to find and extract URLs from the provided email text,
    including http, https, www, and naked domain formats.
    
    Args:
        email_body (str): The body text of the email to analyze
        
    Returns:
        List[str]: A list of all URLs found in the email body
        
    Example:
        >>> urls = extract_urls_from_email("Visit our site at https://example.com or www.example.org")
        >>> print(urls)
        ['https://example.com', 'www.example.org']
    """
    url_pattern = r'https?://[^\s<>"]+|www\.[^\s<>"]+|[a-zA-Z0-9-]+\.(?:[a-zA-Z]{2,})(?:/[^\s<>"]*)?'
    return re.findall(url_pattern, email_body)

@tool
def analyze_domain(domain: str) -> Dict[str, Any]:
    """
    Analyze a domain for age, MX records, and reputation.
    
    This function performs WHOIS lookups and DNS queries to gather information
    about a domain's age, email infrastructure, and registration details.
    
    Args:
        domain (str): The domain to analyze (e.g., "example.com")
        
    Returns:
        Dict[str, Any]: A dictionary containing:
            - domain: The domain that was analyzed
            - age_days: Age of the domain in days (or None if unavailable)
            - has_valid_mx_records: Boolean indicating if valid MX records exist
            - creation_date: String representation of when the domain was created
            - registrar: The domain registrar, if available
            - error: Error message if analysis failed
        
    Example:
        >>> domain_info = analyze_domain("google.com")
        >>> print(f"Domain age: {domain_info['age_days']} days")
    """
    try:
        # Get data about the domain from WHOIS
        domain_info = whois.whois(domain)

        # Get the creation date and try to parse it
        creation_date = domain_info.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]

        # Calculate the domain age
        domain_age = (datetime.now() - creation_date).days if creation_date else None

        # Check for MX domain records which indicates if mail is setup
        has_mx = False
        try:
            mx_records = dns.resolver.resolve(domain, 'MX')
            has_mx = len(mx_records) > 0
        except:
            pass
            
        return {
            "domain": domain,
            "age_days": domain_age,
            "has_valid_mx_records": has_mx,
            "creation_date": str(creation_date) if creation_date else "Unknown",
            "registrar": domain_info.registrar if hasattr(domain_info, 'registrar') else "Unknown"
        }
    except Exception as e:
        return {"error": str(e), "domain": domain}

@tool
def extract_domain_from_email(email: str) -> str:
    """
    Extract the domain from an email address.
    
    This function uses regex to parse and extract the domain portion
    from a standard email address format (user@domain.com).
    
    Args:
        email (str): The email address to extract the domain from
        
    Returns:
        str: The extracted domain name or an error message if parsing fails
        
    Example:
        >>> domain = extract_domain_from_email("user@example.com")
        >>> print(domain)
        'example.com'
    """
    domain_pattern = r'@([A-Za-z0-9.-]+\.[A-Za-z]{2,})'
    match = re.search(domain_pattern, email)
    return match.group(1) if match else "Invalid email format"

##### Agent domain analyser definition

In [ ]:
class DomainAnalysisResult(BaseModel):
    """Result of domain security analysis for phishing detection."""
    is_suspucious: bool = Field(description="Whether the domain is suspicious")
    suspicious_score: int = Field(description="Suspicion score from 0-10")
    domain_findings: str = Field(dscription="Analysis of the sender domain")
    is_phishing: bool = Field(description="Overall determination if email is phishing")
    
class EmailSecurityState(TypedDict):
    """State tracking for email security analysis process."""
    messages: Annotated[List[BaseMessage], add_messages]
    sender_email: str
    subject: str
    body: str
    remaining_steps: int
    structured_response: str

def setup_domain_analysis_agent(llm: LLM) -> AgentExecutor:
    """
    Create an agent for analyzing email domains and URLs for phishing indicators.
    
    This function configures a ReAct agent that uses domain analysis tools to
    investigate suspicious emails. The agent analyzes sender domains and URLs
    to assess the likelihood of the email being a phishing attempt.
    
    Args:
        llm (BaseLLM): The language model to power the agent's reasoning
        
    Returns:
        AgentExecutor: An agent executor that can analyze email domains
        
    Example:
        >>> model = ChatOpenAI(temperature=0)
        >>> domain_agent = setup_domain_analysis_agent(model)
        >>> result = domain_agent.invoke({
        ...     "sender_email": "support@g00gle.com", 
        ...     "subject": "Account Verification",
        ...     "body": "Click here: http://verify-account.suspicious-site.com"
        ... })
    """
    tools = [extract_urls_from_email, analyze_domain, extract_domain_from_email]
    agent_prompt = PromptTemplate(
        template="""You are a security analyst investigating potential phishing emails.
        Use the provided tools to analyze this email domain
        
        CRITICAL INSTRUCTIONS:
        - NEVER provide both an Action AND a Final Answer in the same response
        - ONLY provide a Final Answer after you've completed ALL necessary analysis
        - ALWAYS follow Thought -> Action -> Action Input -> Observation pattern in sequence
        - FINISH all tool usage BEFORE providing Final Answer
    
        Use the following format EXACTLY:
    
        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take
        Action Input: the input to the action
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question
        
        Begin!
        
        SENDER: {sender_email}
        SUBJECT: {subject}
        BODY: {body}
        
        Analyze the sender domain, extract and check URLs, and provide a comprehensive security assessment.        
        Question: Is this domain suspicious based on sender domain and URLs?
        """,
        input_variables=["sender_email", "subject", "body"]
    )

    agent = create_react_agent(
        llm, 
        tools, 
        prompt=agent_prompt,
        response_format=DomainAnalysisResult,
        state_schema=EmailSecurityState,
        checkpointer=MemorySaver())

    return agent

#### Main analysis function

In [ ]:
@dataclass
class AnalysisResult:
    """Comprehensive analysis results for a potentially phishing email."""
    content_analysis: PhishingAnalysisResult
    domain_analysis: DomainAnalysisResult
    similar_emails: List[SimilarEmail]

def analyze_emails(
    llm: LLM,
    samples: DataFrame, 
    vector_store: Chroma, 
    pbar: tqdm
) -> List[AnalysisResult]:
    """
    Perform comprehensive analysis on a batch of email samples.
    
    This function analyzes each email in the provided DataFrame for phishing indicators
    using content analysis, domain analysis, and similarity matching against known
    phishing emails in the vector store.
    
    Args:
        samples (DataFrame): DataFrame containing email samples with columns:
                           'sender', 'subject', 'body'
        vector_store (Chroma): Vector store containing embedded email documents
                              for similarity comparison
        pbar (tqdm): Progress bar for tracking analysis progress
        
    Returns:
        List[AnalysisResult]: A list of analysis results for each email sample,
                             containing content analysis, domain analysis, and
                             similar phishing emails found
    
    Example:
        >>> from tqdm.auto import tqdm
        >>> pbar = tqdm(total=60)
        >>> results = analyze_emails(email_samples, vector_store, pbar)
        >>> for result in results:
        ...     if result.content_analysis.is_phishing or result.domain_analysis.is_phishing:
        ...         print("Phishing email detected!")
    """
    email_analysis_chain = setup_email_analysis_chain(llm)
    domain_analysis_agent = setup_domain_analysis_agent(llm)
    results: List[AnalysisResult] = []
    sample_len = len(samples)
    progress_bar_update =  60 / sample_len
    index = 1
    
    for _, row in samples.iterrows():
        pbar.set_description(f"Analyzing email content... {index}/{sample_len}")
        content_analysis = analyze_email_content(row, email_analysis_chain)
    
        pbar.set_description(f"Analyzing domain with agent... {index}/{sample_len}")
        domain_analysis_response = domain_analysis_agent.invoke({
            "sender_email": row['sender'],
            "subject": row['subject'],
            "body": row['body'],
            "messages": [],
            "remaining_steps": 10
        },
        config={
            "configurable": {
                "thread_id": "email_analysis_1"
            }
        })
    
        pbar.set_description(f"Find similar with example email... {index}/{sample_len}")
        similar_emails = find_similar_phishing_emails(
            vector_store, 
            row['subject'], 
            row['body']
        )

        results.append(
            AnalysisResult(
                content_analysis, 
                domain_analysis_response['structured_response'], 
                similar_emails
            )
        )
        pbar.update(progress_bar_update)
        index += 1

    return results

### Training Materials Generation

In [ ]:
# Langchain has an issue with properly parsing the Dict type
# so the custom KeyValuePair type was created to be able to parse correctly the Q&A section.
TKey = TypeVar("TKey")
TValue = TypeVar("TValue")
class KeyValuePair(BaseModel, Generic[TKey, TValue]):
    """A generic key-value pair structure."""
    key: TKey
    value: TValue
    
class TrainingContent(BaseModel):
    """Structured phishing awareness training content model."""
    introduction: str = Field(description="Introduction to phishing")
    techniques_explained: List[str] = Field(description="Common phishing techniques explained")
    examples: List[str] = Field(description="Real-world examples")
    identification_guide: List[str] = Field(description="Steps to identify phishing")
    action_items: List[str] = Field(description="What to do if you suspect phishing")
    quiz_questions: List[KeyValuePair[str, List[str]]]  = Field(description="Quiz questions with answers in dictionary format")
    
def setup_training_materials_generation_chain(llm: LLM) -> Chain:
    """
    Creates a chain for generating comprehensive phishing awareness training materials.
    
    This function configures an LLM chain that generates structured training content
    based on provided phishing techniques. The output includes educational sections,
    real-world examples, identification guidelines, recommended actions, and quiz
    questions with answers.
    
    Args:
        llm (BaseLLM): The language model to use for generating training content
    
    Returns:
        LLMChain: A chain that takes phishing techniques as input and returns
                 structured training content
    
    Example:
        >>> model = ChatOpenAI(temperature=0.7)
        >>> training_chain = setup_training_materials_generation_chain(model)
        >>> techniques = ["Impersonation", "Urgency", "URL manipulation", "Attachment phishing"]
        >>> training_material = training_chain.invoke({"techniques": ", ".join(techniques)})
        >>> print(f"Training contains {len(training_material.quiz_questions)} quiz questions")
    """
    training_prompt = PromptTemplate(
        input_variables=["techniques"],
        template="""
        Create a comprehensive phishing awareness training guide based on these techniques:
        {techniques}
        
        Include the following sections:
        1. Introduction to Phishing
        2. Explanation of Phishing Techniques
        3. Real-World Examples (provide examples for each technique)
        4. How to Identify Phishing Attempts (specific indicators)
        5. What to Do If You Suspect Phishing
        6. Knowledge Check Questions (3-5 quiz questions with answers)

        For quiz_questions, provide a dictionary where:
        - Each key is a complete question
        - Each value is a list of possible answers
        
        Ensure your response is valid JSON that can be parsed into the TrainingContent model.
        """
    )
    
    training_chain = (
        training_prompt
        | llm.with_structured_output(TrainingContent)
    )

    return training_chain

In [ ]:
def generate_training_materials(
    llm: LLM, 
    phishing_examples: List[AnalysisResult]
) -> TrainingContent:
    """
    Generate educational training materials from analyzed phishing examples.
    
    This function extracts manipulation techniques from a collection of analyzed
    phishing examples and uses them to generate comprehensive training materials.
    The generated content includes explanations, examples, identification guidelines,
    and quiz questions focused on the specific phishing techniques found.
    
    Args:
        phishing_examples (List[AnalysisResult]): A list of analyzed phishing emails
                                                 containing detection results
    
    Returns:
        TrainingContent: Structured training content with sections including:
                        - Introduction to phishing
                        - Explanations of detected techniques
                        - Real-world examples
                        - Identification guidelines
                        - Recommended actions
                        - Quiz questions with answers
    
    Example:
        >>> analysis_results = analyze_emails(email_samples, vector_store, progress_bar)
        >>> training_materials = generate_training_materials(analysis_results)
        >>> print(f"Generated {len(training_materials.techniques_explained)} technique explanations")
    """
    training_generation_chain = setup_training_materials_generation_chain(llm)
    
    techniques = []
    for example in phishing_examples:
        # Make sure that manipulation techniques are set as sometimes LLM can make an error and don't produce any
        if example != None and example.content_analysis != None and example.content_analysis.manipulation_techniques != None:
            techniques.extend(example.content_analysis.manipulation_techniques)
    
    unique_techniques = list(set(techniques))
    training_material = training_generation_chain.invoke({"techniques": unique_techniques})
    
    return training_material

## System Execution

In [ ]:
# Number of samples to be used for Vector DB creation
db_size = 5

# Number of samples to be used for classification, indicates the count to take from each class (so in our case 2 * 2 = 4)
# As there are some restriction on free API tier it is set to low value
sample_size = 2

# Split dataset into two data frames one for phishing and one for legit email samples 
phishing_samples = df[df['label'] == 1]
legit_samples = df[df['label'] == 0]

# Create testing sample by sampling from both data frames
testing_samples = pd.concat([phishing_samples.sample(sample_size), legit_samples.sample(sample_size)])

# Split into data and labels
X, y = testing_samples.drop("label", axis=1), testing_samples["label"]

# Print size of datasets 
print(f"Full Data set: {df.shape[0]} emails")
print(f"Sampled Data set: {X.shape[0]} emails")

In [ ]:
# Create Google GenAI LLM and Embedding models
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
embedding = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

In [ ]:
# Create progress bar so it looks nicer
pbar = tqdm(total=100, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}')

db_input = phishing_samples.sample(db_size)
pbar.set_description("Building vector database...")
vector_store = create_email_vector_store(db_input, embedding)
pbar.update(20)

pbar.set_description("Analyzing emails...")
analysis_results  = analyze_emails(llm, X, vector_store, pbar)

phising_examples = [result for result in analysis_results if result.content_analysis != None and result.content_analysis.is_phishing]
pbar.set_description("Generating training materials...")
training_material_raw = generate_training_materials(llm, analysis_results)
pbar.update(20)
pbar.close()

In [ ]:
# Display accuracy
tp = [1 for i, result in enumerate(analysis_results) if result.content_analysis != None and result.content_analysis.is_phishing == y.iloc[i]]
tn = [1 for i, result in enumerate(analysis_results) if result.content_analysis != None and result.content_analysis.is_phishing != y.iloc[i]]
accuracy = (sum(tp)+sum(tn)) / len(X) * 100
print(f"Correctly classified examples: {accuracy}%")

In [ ]:
# Prepare analysis report
analysis_report = """
# Phishing Analysis Report
"""

for i, result in enumerate(analysis_results, 1):
    if result.content_analysis == None:
        continue
        
    phishing_score = result.content_analysis.phishing_score
    is_phishing = "Yes" if result.content_analysis.is_phishing else "No"
    suspicious_elements = ", ".join(result.content_analysis.suspicious_elements) if len(result.content_analysis.suspicious_elements) > 0 else "None"
    manipulation_techniques = ", ".join(result.content_analysis.manipulation_techniques) if len(result.content_analysis.manipulation_techniques) > 0 else "None"

    is_domain_suspucious = "Yes" if result.domain_analysis.is_suspucious else "No"
    domain_suspicious_score = result.domain_analysis.suspicious_score
    domain_findings = result.domain_analysis.domain_findings
    is_phishing_domain = "Yes" if result.domain_analysis.is_phishing else "No"
    
    result_report_part = f"""
## Analysis {i}

### Content Analysis
**Email Title**: {X.iloc[i-1]['subject']}

**Email Sender**: {X.iloc[i-1]['sender']}

**Phishing Score**: {result.content_analysis.phishing_score}/10

**Is Phishing**: {is_phishing}

**Suspicious Elements**: {suspicious_elements}

**Manipulation Techniques**: {manipulation_techniques}

### Domains Analysis
**Suspicious Score**: {domain_suspicious_score}

**Is Suspicious**: {is_domain_suspucious}

**Is Phishing**: {is_phishing_domain}

**Findings**: {domain_findings}
"""
    if result.content_analysis.is_phishing:
        result_report_part += "### Similar Cases"
        for j, case in enumerate(result.similar_emails, 1):
            result_report_part += f"""
#### Case {j}

**Sender**: {case.metadata['sender']}

**Content**: {case.content}
"""

    result_report_part += """
### Recommendations:
"""
    for j, rec in enumerate(result.content_analysis.recommendations, 1):
        result_report_part += f"{j}. {rec}\n"

    analysis_report += result_report_part


display(Markdown(analysis_report))

In [ ]:
#Prepare training material
training_material = f"""
# Introduction
{training_material_raw.introduction}

# Explained Phishing Techniques
"""
for j, technique in enumerate(training_material_raw.techniques_explained, 1):
    training_material += f"{j}. {technique}\n"

training_material += """
# Examples
"""
for j, example in enumerate(training_material_raw.examples, 1):
    training_material += f"{j}. {example}\n"

training_material += """
# Identification Steps
"""
for j, step in enumerate(training_material_raw.identification_guide, 1):
    training_material += f"{j}. {step}\n"

training_material += """
# Action Items
"""
for j, action in enumerate(training_material_raw.action_items, 1):
    training_material += f"{j}. {action}\n"

training_material += """
# Q&A
"""
for j, question in enumerate(training_material_raw.quiz_questions, 1):
    training_material += f"## Question {j}. {question.key}\n"
    for k, answer in enumerate(question.value, 1):
        training_material += f"{k}. {answer}\n"

display(Markdown(training_material))